# 固定 S/Z 监督：真实训练域无拟合预检已完成

一次exec `56431` exit0，原3014/20741、85→87、完整目标/tensor/scaler/驻留范围及三seed配对初始化通过。真实梯度、optimizer、fit/account和评价对象仍为0。

[冻结计划](plan.json) · [预检汇总](preflight/summary.json) · [执行收据](preflight/execution_receipt.json) · [完整中文结论](conclusion.md)


## 原生 Qlib 入口与分阶段对象边界

获准的预检会执行 `qlib.init` → 在无全局写入 Recorder 时 `R.get_recorder`/`Recorder.load_object` 只读已核验白名单 → `DatasetH.prepare(col_set=DataHandler.CS_RAW)` → 严格日期/原始8字段证明后 `DataHandlerLP.from_df`/`DatasetH`（85→87列）→ `config(dump_all=True,recursive=True)` → `with R.start` 保存与直接 prepare 读回。禁止 `setup_data` 回载和先加载全时期对象再裁剪。

原PG instrument是SOXL_SOXX_CASH；C来源是SOXL_SOXX。严格验证完整3014日期顺序与原目标后才按位置赋值，不做inner join。C Recorder artifact `labels/fixed_H5_components_S_Z` 的实际列组是 `original`(8) 和 `fixed_day_night_label`(11)，只取后者S/Z。

正式运行先封存六模型SHA，才读取原946行评价Dataset和944个账户区间；预测不要求监督列。关闭全局R.start后，以同一明确Recorder低层start_run/end_run包裹评价与原生回放，避免source reader/qlib.init冲突。


In [1]:
import os, json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
CARD = ROOT / "research/stage1_day_night_supervision_20261018"
plan = json.loads((CARD / "plan.json").read_text())
api_chain = ["qlib.init", "R.get_recorder / Recorder.load_object(exact verified whitelist)", "DatasetH.prepare(DataHandler.CS_RAW)", "DataHandlerLP.from_df -> DatasetH.config(dump_all=True, recursive=True)", "R.start / Recorder.save_objects / direct prepare roundtrip"]
prefit = os.environ.get("DAY_NIGHT_SUPERVISION_PREFLIGHT") == "1"
execute = os.environ.get("DAY_NIGHT_SUPERVISION_EXECUTE") == "1"
if prefit and execute:
    raise RuntimeError("Preflight and fit stages must be separately approved")
if prefit or execute:
    from src.workflow.day_night_supervision import prepare, run
    display({"approved_workflow_API_chain": api_chain, "phase": "preflight" if prefit else "fit_then_accounts"})
    result = (prepare if prefit else run)(root=ROOT, card=CARD)
    display({k: result.get(k) for k in ["decision", "recorder_id", "new_fits", "new_native_accounts", "gates", "nas_evidence"]})
else:
    display({"status": "implementation_only", "future_authorized_API_chain": api_chain, "real_source_objects_loaded": 0, "fits": 0, "accounts": 0})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'approved_workflow_API_chain': ['qlib.init',
  'R.get_recorder / Recorder.load_object(exact verified whitelist)',
  'DatasetH.prepare(DataHandler.CS_RAW)',
  'DataHandlerLP.from_df -> DatasetH.config(dump_all=True, recursive=True)',
  'R.start / Recorder.save_objects / direct prepare roundtrip'],
 'phase': 'preflight'}

2026/10/18 19:57:39 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[41137:MainThread](2026-10-18 19:57:39,573) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[41137:MainThread](2026-10-18 19:57:39,575) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


...

[41137:MainThread](2026-10-18 19:57:39,928) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_day_night_supervision_20261018_preflight.


[41137:MainThread](2026-10-18 19:57:39,929) INFO - qlib.workflow - [exp.py:258] - Experiment 995544772562124587 starts running ...


[41137:MainThread](2026-10-18 19:57:40,016) INFO - qlib.workflow - [recorder.py:345] - Recorder 737188ea81cb42dab51327e2c3265911 starts running under Experiment 995544772562124587 ...


[41137:MainThread](2026-10-18 19:57:41,235) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'eligible_for_formal_design_review',
 'recorder_id': '737188ea81cb42dab51327e2c3265911',
 'new_fits': 0,
 'new_native_accounts': None,
 'gates': None,
 'nas_evidence': {'sha256': '2d17e38105552b6760ce5d32fff3a1e30c471ed04ab2fdfe99e0ac81490943e7',
  'size_bytes': 2791752,
  'media_type': 'application/gzip',
  'storage_uri': 'sha256/2d/17/2d17e38105552b6760ce5d32fff3a1e30c471ed04ab2fdfe99e0ac81490943e7',
  'uploaded_by': 'soxl-minute-research',
  'created_at': '2026-10-18 19:57:45.565316+08',
  'duplicate': False,
  'nas_path': '/volume1/docker/yanxuan/registry/research-artifacts/2d/17/2d17e38105552b6760ce5d32fff3a1e30c471ed04ab2fdfe99e0ac81490943e7',
  'recorder_id': '737188ea81cb42dab51327e2c3265911',
  'experiment_id': '995544772562124587',
  'archive_root': 'qlib_recorder'}}

## 配对、费用与证据含义

两套目标只在获准训练域预检中各拟合一次float64均值/sample std(ddof=1)，标准化后仅一次float32转换；六fit复用同一冻结常数，不做clip/epsilon救援。原combined loss后才加0.02×half-MSE；仍一次combined backward、全982参数global clip2、AdamW。零head首步直接encoder梯度为0，但head梯度可经globalclip改变private更新，不能将结果归于单纯encoder语义。

原目标还是H1 open→next-open allocation、H5 tail/alpha/cash与原共同成熟筛选。S/Z来自已独评接受的July24分配调整研究价格平面，不是现金股利账本、原daily-reset alpha或实际09:47成交标签。

账户预算为2个等权ensemble×3个费用=6条独立自身状态政策路径/6新native，另复用15本参考，共21比较与15预先固定CI。所有成本/seed保留，MSE/AUC不是账户执行门。0新市场来源查询不等于完全不读账户价格：六模型屏障后获准解码固定raw_points/market，并生成/读取本卡Qlib execution provider；不构造S/Z评价标签。


In [2]:
display(Markdown((CARD / "conclusion.md").read_text()))


# S/Z 监督：无拟合预检完成

只完成获批的训练域来源/严格日期与instrument映射、85→87列保真、原tensor/scaler与初始化核验。原3014/20741和全部目标/时钟未变；两套S/Z缩放常数仅拟合一次，未评判可预测性。

0真实optimizer step、0fit/account、0评价对象或新价格查询。正式6fits及6成本独立账户仍需最终设计评审与单独owner GO；不因标签或软件检查通过而晋升。


# S/Z 配对监督：无拟合预检已验收，正式执行已批准

唯一真实预检 56431 exit 0。主训练保持原 3014 行与全部 85 列，只追加 S/Z 两列至 87；辅助训练 20741×20 未变。原目标、时钟、11 个输入张量及原 scaler 精确一致，所有实际 resident frames 都限于训练域。两套 S/Z 缩放统计各计算一次，并固定原始值、完整索引与标准化值哈希。

三个 seed 的原 973 参数子集、配对 982 参数、原输出及 RNG 均通过初始化核验。Recorder `737188ea81cb42dab51327e2c3265911` 的 15 个 artifacts 已归档 NAS，唯一冷恢复 87112 exit 0；冷读只应用已冻结统计核对哈希，没有重拟合尺度。独立评估接受预检证据并签署正式设计评审。

截至此处真实模型拟合、optimizer step、政策路径、账户与评价对象读取仍为 0；没有预测能力、市场分窗或经济结果，PIT 和独立前向资格仍未通过，未晋升。

已单独批准固定六次 300-epoch 拟合；全部六个终态模型及身份封存后才读取评价对象，随后完成 B/C×0/2/5bp 六条独立自有状态政策路径和六个原生账户、十五个固定参考及十五项配对区间。训练分数不筛选或取消账户；不改 seed、权重、窗口、控制器或现役 release。总 Goal 继续。
